### Match nodes to networks

*This script was used to solve a problem when plotting results in in the IS-RSA Analysis: The data we received from preprocessing are in the order of the Yeo 17 parcellation, but that differs from the Yeo 7 parcellation. However, I need the Yeo 7 parcellation for the node-to-network assignment. Here it was figured out, which node name from the Yeo 7 parcellation (or index) belongs to the Yeo 17 parcellation.*

In [2]:
import pandas as pd
import os 

print(os.getcwd())
os.chdir(r"C:\Users\jop86ib\Documents\Paper 4\Code\Test Stuff\Node_network_mapping")
print(os.getcwd())

C:\Users\jop86ib\Documents\Paper 4\Code\Test Stuff
C:\Users\jop86ib\Documents\Paper 4\Code\Test Stuff\Node_network_mapping


#### 1. Import Excel files 
*(from https://github.com/ThomasYeoLab/CBIG/tree/master/stable_projects/brain_parcellation/Yan2023_homotopic/parcellations/MNI/centroid_coordinates)*

In [5]:
yeo_7_order = pd.read_csv('Parcels_Yeo2011_7Networks_FSLMNI152_2mm.csv')
yeo_17_order = pd.read_csv('Parcels_Yeo2011_17Networks_FSLMNI152_2mm.csv')

In [7]:
print(yeo_7_order.columns)
print(yeo_17_order.columns)

Index(['ROI Label', 'ROI Name', 'R', 'A', 'S'], dtype='object')
Index(['ROI Label', 'ROI Name', 'R', 'A', 'S'], dtype='object')


#### 2. Check whether all triplets (coordinates) in the Yeo 17 list also exist in the yeo 7 list

In [22]:
# Extract all coordinates as sets of triplets
coords_7 = set(tuple(x) for x in yeo_7_order[['R', 'A', 'S']].values)
coords_17 = set(tuple(x) for x in yeo_17_order[['R', 'A', 'S']].values)

# Comparison: Are all coordinates from yeo_7_order also in yeo_17_order?
missing_coords = coords_7 - coords_17

if len(missing_coords) == 0:
    print("All coordinate triplets from yeo_7_order also exist in yeo_17_order.")
else:
    print(f"{len(missing_coords)} Coordinate triplets from yeo_7_order are missing in yeo_17_order:")
    for triplet in missing_coords:
        print(triplet)


All coordinate triplets from yeo_7_order also exist in yeo_17_order.


#### 3. Find out which node (index) in Yeo 7 order belongs to Yeo 17 order --> which is the order that my data is in

In [24]:
matches = []

for idx_17, row_17 in yeo_17_order.iterrows():
    coords_17 = (row_17['R'], row_17['A'], row_17['S'])

    match_row = yeo_7_order[
        (yeo_7_order['R'] == coords_17[0]) &
        (yeo_7_order['A'] == coords_17[1]) &
        (yeo_7_order['S'] == coords_17[2])
    ]

    if not match_row.empty:
        match = match_row.iloc[0]
        matches.append({
            'yeo_17_index': idx_17,
            'yeo_17_name': row_17['ROI Name'],
            'yeo_17_R': row_17['R'],
            'yeo_17_A': row_17['A'],
            'yeo_17_S': row_17['S'],
            'yeo_7_index': match.name,
            'yeo_7_name': match['ROI Name'],
            'yeo_7_R': match['R'],
            'yeo_7_A': match['A'],
            'yeo_7_S': match['S']
        })
    else:
        matches.append({
            'yeo_17_index': idx_17,
            'yeo_17_name': row_17['ROI Name'],
            'yeo_17_R': row_17['R'],
            'yeo_17_A': row_17['A'],
            'yeo_17_S': row_17['S'],
            'yeo_7_index': None,
            'yeo_7_name': None,
            'yeo_7_R': None,
            'yeo_7_A': None,
            'yeo_7_S': None
        })

# Create dataframe
matches_df = pd.DataFrame(matches)

# Save as csv
matches_df.to_csv('yeo_17_to_7_matches_with_coords.csv', index=False)

print("File 'yeo_17_to_7_matches_with_coords.csv' successfully saved!")


File 'yeo_17_to_7_matches_with_coords.csv' successfully saved!
